---
## Paso 7: Análisis y Discusión

Respuestas a las 9 preguntas del PDF basadas en los resultados experimentales obtenidos.


In [ ]:
print("=" * 70)
print("  ANÁLISIS Y DISCUSIÓN — 9 PREGUNTAS")
print("=" * 70)

preguntas = {

1: ("¿Qué ventajas ofrece Flax NNX respecto a implementar modelos directamente en JAX?",
"""
Flax NNX proporciona una capa de abstracción orientada a objetos sobre JAX puro que
simplifica significativamente la construcción de modelos. Las ventajas observadas:

• Gestión automática del estado del modelo: NNX administra internamente los parámetros,
  estadísticas de BatchNorm y estados de Dropout mediante nnx.Module, eliminando la
  necesidad de manejar manualmente pytrees de parámetros.

• Componentes reutilizables listos: nnx.Conv, nnx.Linear, nnx.BatchNorm y nnx.Dropout
  encapsulan inicialización, forward pass y actualización de estado, reduciendo código
  repetitivo sustancialmente.

• Integración natural con jax.jit mediante @nnx.jit: el decorador maneja la separación
  automática entre estado mutable y funciones puras requerida por JAX.

• En JAX puro habría sido necesario gestionar manualmente los dicts de parámetros,
  aplicar jax.grad sobre funciones que reciben parámetros como argumento, y manejar
  el estado de BatchNorm separado de los pesos. Flax NNX abstrae toda esa complejidad."""),

2: ("¿Qué beneficios aporta la compilación mediante jax.jit?",
"""
Los beneficios observados directamente durante el entrenamiento:

• Reducción drástica de tiempo tras la primera época: la época 1 tomó ~48s (compilación
  XLA), mientras las épocas siguientes tomaron ~9-12s — una mejora de ~4-5×.

• Fusión de operaciones: XLA fusiona operaciones matemáticas consecutivas (Conv + BN +
  ReLU) en un único kernel de GPU, reduciendo transferencias de memoria intermedias.

• Throughput sostenido de 225-294 img/s en T4 durante todo el entrenamiento.

• Eliminación del overhead de Python en el bucle interno: una vez compilado, el paso
  de entrenamiento se ejecuta completamente en GPU sin intervención del intérprete."""),

3: ("¿Qué hiperparámetro tuvo mayor impacto sobre la exactitud final?",
"""
El número de épocas de entrenamiento tuvo el mayor impacto positivo en exactitud:

• Modelo base (20 épocas):  exactitud prueba = 70.23%
• Config E    (30 épocas):  exactitud prueba = 75.16%  (+4.93 pp)

El learning rate tuvo el mayor impacto negativo cuando era inadecuado:
• lr=1e-2: exactitud prueba = 18.42%  (modelo que nunca convergió)
• lr=1e-3: exactitud prueba = 68.91%  (diferencia de +50.49 pp)

Seguido por el tamaño de lote:
• batch=32:  70.72%  vs  batch=128: 48.76%  (diferencia de 21.96 pp)"""),

4: ("¿Qué hiperparámetro tuvo mayor impacto sobre el tiempo de entrenamiento?",
"""
El batch_size fue el hiperparámetro con mayor impacto sobre el tiempo por época:

• batch=16:  10.8 s/época  (más iteraciones por época, más overhead)
• batch=32:   9.4 s/época
• batch=128: 10.9 s/época  (lotes más grandes, más tiempo por paso GPU)

El número de filtros/neuronas tuvo impacto moderado:
• f32-n128:   10.5 s/época  (1M params)
• f128-n512:  11.2 s/época  (16M params)  — solo +0.7s pese a 16× más parámetros,
  gracias a la eficiencia del paralelismo en GPU."""),

5: ("¿Cómo afectó el tamaño de lote al rendimiento observado?",
"""
Resultados experimento 6.1:

• batch=32:  Acc.Test=70.72%, throughput=286 img/s  — MEJOR exactitud
• batch=64:  Acc.Test=60.00%, throughput=282 img/s
• batch=128: Acc.Test=48.76%, throughput=247 img/s  — PEOR exactitud

Análisis: lotes más pequeños producen gradientes con mayor varianza (ruido), lo que
actúa como regularizador implícito y ayuda a escapar mínimos locales. Con batch=128,
el modelo converge a soluciones más planas pero menos generalizables dado el tamaño
reducido del dataset (3,846 imágenes). En datasets más grandes, batch=128 normalmente
produce mejor throughput sin sacrificar exactitud significativamente."""),

6: ("¿Qué diferencias encontró entre float32, float16 y bfloat16?",
"""
Resultados experimento de precisión numérica:

dtype     Acc.Test  Estabilidad  T/época  Memoria
float32   75.16%    ✓ Estable    9.7s     8.4 MB   ← mayor exactitud
float16   73.85%    ✓ Estable    9.9s     4.2 MB   ← mejor balance
bfloat16  66.94%    ✓ Estable   10.1s     4.2 MB   ← menor exactitud

• Los 3 formatos fueron estables en T4 — la GPU T4 tiene soporte nativo para float16.
• float32 logró la mayor exactitud (+1.31 pp vs float16, +8.22 pp vs bfloat16).
• float16 reduce la memoria a la mitad con pérdida mínima de exactitud (-1.31 pp).
• bfloat16 fue el menos preciso pese a tener mayor rango dinámico que float16 —
  su menor precisión de mantisa (7 bits) afectó la calidad de los gradientes."""),

7: ("¿Cuál configuración produjo el mejor balance entre rendimiento y exactitud?",
"""
La configuración E con float32 produjo el mejor balance global:

  batch=32 · lr=1e-3 · filtros=32→64→128 · neuronas=256 · 30 épocas · float32

  Exactitud prueba : 75.16%
  Throughput       : 294 img/s
  Tiempo/época     : 9.1 s
  Memoria          : 8.4 MB

Si se priorizara eficiencia de memoria sobre exactitud:
  float16 con config E: ~73.85% acc prueba, 4.2 MB — ahorra 50% de memoria
  con solo -1.31 pp de exactitud."""),

8: ("¿Qué limitaciones encontró al utilizar Google Colab?",
"""
• Tiempo de sesión: Colab desconecta la sesión tras ~90 min de inactividad o ~12 hrs
  de uso continuo, lo que obligó a re-ejecutar todo el pipeline desde cero en ocasiones.

• RAM limitada: la sesión gratuita dispone de ~12 GB de RAM CPU. Pipelines con
  batch_size=16 y prefetch AUTOTUNE saturaron la memoria en algunos experimentos.

• GPU no garantizada: en horas pico no siempre se asignó T4, lo que varía los tiempos.

• Sin persistencia de variables: al reiniciar la sesión se pierden todas las variables
  Python, modelos entrenados y resultados — sin checkpoint propio, hay que reentrenar.

• Disco efímero: los archivos en /content desaparecen al reiniciar. Se requiere
  Google Drive o re-descarga del dataset en cada sesión."""),

9: ("¿Qué mejoras futuras propondría para aumentar el desempeño del modelo?",
"""
1. Aumento de datos más agresivo: rotaciones ±30°, zoom, recortes aleatorios y
   mixup/cutmix podrían mejorar la generalización significativamente dado el
   tamaño reducido del dataset (3,846 imágenes para 12 clases).

2. Transfer learning: usar una red preentrenada (ResNet, EfficientNet, ViT) como
   feature extractor y solo entrenar las capas finales — esperaría >90% de exactitud
   con el mismo dataset.

3. Learning rate scheduling más sofisticado: warm-up + cosine annealing con reinicios
   (SGDR) para escapar mejor los mínimos locales en las épocas finales.

4. Regularización adicional: L2 weight decay en las capas densas y label smoothing
   en la función de pérdida para reducir sobreconfianza del modelo.

5. Arquitectura más profunda con residual connections (ResNet-style): permitiría
   entrenar redes más profundas sin degradación del gradiente.

6. Aceleración con GPU dedicada (A100/V100): el throughput podría escalar de
   ~294 img/s a >2,000 img/s con bfloat16 y batch_size=256."""),
}

for num, (pregunta, respuesta) in preguntas.items():
    print(f"\n{'─'*70}")
    print(f"  Pregunta {num}: {pregunta}")
    print(f"{'─'*70}")
    print(respuesta)

print(f"\n{'='*70}")
print("  Análisis y Discusión completado — 9/9 preguntas respondidas")
print(f"{'='*70}")


### Resumen ejecutivo del proyecto

In [ ]:
print("=" * 65)
print("  RESUMEN EJECUTIVO — Tarea 2")
print("  Aprendizaje Distribuido para Computer Vision")
print("=" * 65)
print()
print("  Dataset      : Wonders of the World")
print("  Clases       : 12")
print("  Imágenes     : 3,846  (train 70% / val 15% / test 15%)")
print("  Acelerador   : Tesla T4 — CUDA 13.0")
print("  Framework    : JAX 0.10.1 + Flax NNX 0.12.7 + Optax 0.2.8")
print()
print("  Arquitectura óptima:")
print("    Conv(32)→BN→ReLU→MaxPool")
print("    Conv(64)→BN→ReLU→MaxPool")
print("    Conv(128)→BN→ReLU→MaxPool")
print("    Dense(256)→Dropout(0.4)→Dense(12)")
print("    Parámetros: 2,194,638")
print()
print("  Hiperparámetros óptimos:")
print("    batch_size : 32")
print("    lr         : 1e-3  (cosine decay)")
print("    épocas     : 30")
print("    dtype      : float32")
print()
print("  Resultados finales:")
print("    Exactitud entrenamiento : 68.60%")
print("    Exactitud validación    : 76.48%")
print("    Exactitud prueba        : 75.16%")
print("    Throughput              : 294 img/s")
print("    Tiempo/época            : 9.1 s")
print()
print("  Tarea 2 completada — todos los requisitos cubiertos.")
print("=" * 65)
